In [ ]:
# In[1]: Read Documents and Fixed-Size Batching
import csv
import json
import os
import sys

# Increase CSV field size limit for large text fields
csv.field_size_limit(sys.maxsize)

# Read all documents
csv_path = "/home/nena-meijer/PyCharmMiscProject/dataset/VWS_subset/6-VWS_documents_NER_nl_labels.csv"
print(f"Reading CSV from {csv_path}")
docs = []
with open(csv_path, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    for idx, row in enumerate(reader, start=1):
        docs.append({'id': row['document_id'], 'text': row['document_text']})
        if idx % 500 == 0:
            print(f"Loaded {idx} documents")
print(f"Total documents loaded: {len(docs)}")

# Split into fixed-size batches of 100 documents each
BATCH_SIZE = 6
batches = [docs[i:i + BATCH_SIZE] for i in range(0, len(docs), BATCH_SIZE)]
print(f"Created {len(batches)} batches of up to {BATCH_SIZE} docs each.")

# In[2]: Save Batches to Files
output_dir = "batches"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
for idx, batch in enumerate(batches, start=1):
    batch_path = os.path.join(output_dir, f"batch_{idx}.json")
    with open(batch_path, 'w', encoding='utf-8') as bf:
        json.dump({'documents': batch}, bf, ensure_ascii=False, indent=2)
    print(f"Saved batch {idx}/{len(batches)} with {len(batch)} docs to {batch_path}")

In [10]:
API_key=""

In [8]:
documentids = ["134-3004", "117-443", "130-1544", "130-1545", "130-1546", "130-1658", "134-1191", "130-2239", "117-1054", "133-2263", "133-2264", "195-353"]

In [ ]:
import os
import json
import re
from google import genai
from google.genai import types

# Setup Gemini client and model
client = genai.Client(api_key=API_key)
MODEL_NAME = "models/gemini-2.0-flash"
print(f"Using model: {MODEL_NAME} for API generation")

# Instruction template
INSTRUCTIONS = (
    "Je krijgt een JSON-array met documenten met betrekking tot de coronacrisis in het Nederlands of Engels.\n"
    "Geef de resultaten terug in het Nederlands. Voor elk document extraheer je de volgende informatie:\n"
    "1. events: alle events met betrekking tot de coronacrisis en de besluitvorming rondom de coronacrisis. Geef de bijbehorende datums van de events in het format: YYYY-MM-DD. Een 'event' is een gebeurtenis die directe invloed heeft op de reactie op de coronacrisis, beleidsbeslissingen, of significante veranderingen in de situatie. Voorbeelden zijn: aankondigingen van nieuwe maatregelen, wetenschappelijke publicaties die het beleid beïnvloeden, belangrijke politieke debatten, en wijzigingen in test- of vaccinatiestrategieën.\n"
    "De relatieve datums (morgen, gister, vandaag, volgende week, etc.) moeten ook worden omgezet naar het formaat YYYY-MM-DD.  Het is cruciaal dat alle datums, inclusief relatieve datums en verwijzingen naar dagen van de week, correct worden omgezet naar het YYYY-MM-DD formaat. Gebruik de context van het document om de juiste datum te bepalen. Indien de documenten datums bevatten gerelateerd aan 'vandaag' interpreteer dit als de datum van analyse, welke vandaag: 2020-06-22 is. Alle datums, absoluut en relatief, moeten geretourneerd worden in het format YYYY-MM-DD. Er mogen geen uitzonderingen zijn.\n"
    "Geef een beschrijving van het event die voldoende context bevat om te begrijpen wat er gebeurt en waarom het relevant is voor de coronacrisis.  Streef naar een beschrijving van minimaal 20 woorden, maar wees niet bang om meer te gebruiken als dat nodig is om de context volledig te omvatten.  Bij het beschrijven van een event, probeer de volgende vragen te beantwoorden: Wie was erbij betrokken? Wat is er precies gebeurd? Waar vond het plaats (indien relevant)? Waarom is dit event belangrijk in de context van de coronacrisis?\n"
    "De beschrijving van het event moet gebaseerd zijn op de omliggende tekst in het document. Probeer zinnen of fragmenten uit het document te gebruiken om de beschrijving zo accuraat en informatief mogelijk te maken. Vermijd vage of onvolledige beschrijvingen zoals 'Gesprek gehad' of 'Beslissing genomen'. Geef altijd de belangrijkste details.\n"
    "Hier zijn een paar voorbeelden:\n"
    "INPUT DOCUMENT: 'Vandaag heeft het RIVM nieuwe richtlijnen gepubliceerd voor het gebruik van mondkapjes in het openbaar vervoer.  Morgen zal de minister een persconferentie geven om de details toe te lichten.'\n"
    "OUTPUT: {\"document_id\": \"voorbeeld-1\", \"events\": [{\"event\": \"Het RIVM publiceert nieuwe richtlijnen voor het gebruik van mondkapjes in het openbaar vervoer, waardoor reizigers en vervoerders nieuwe regels moeten volgen.\", \"date\": \"2023-10-27\"}, {\"event\": \"De minister geeft een persconferentie over de nieuwe RIVM-richtlijnen voor mondkapjes in het openbaar vervoer, waarin de details en de achtergrond van de regels worden toegelicht.\", \"date\": \"2023-10-28\"}]}\n"
    "Stuur als output één JSON-object met een \"results\"-array.\n"
    "Neem per document het document_id mee in de response.\n"
    "Lever uitsluitend geldige JSON zonder extra markdown‑fences, zonder trailing commas, met alle strings correct geescaped.\n"
)

# Batch directory and range
BATCH_DIR = "/home/nena-meijer/PyCharmMiscProject/information_extraction/batches"
BATCH_START = 2001
BATCH_END = 3813  # inclusive

aggregated_results = []

for i in range(BATCH_START, BATCH_END + 1):
    batch_path = os.path.join(BATCH_DIR, f"batch_{i}.json")
    if not os.path.isfile(batch_path):
        print(f"Batch {i}: bestand niet gevonden, overslaan...")
        continue

    with open(batch_path, 'r', encoding='utf-8') as f:
        try:
            batch = json.load(f)
        except json.JSONDecodeError as e:
            print(f"Batch {i}: JSON decode error bij het inlezen van bestand: {e}")
            aggregated_results.append({'batch': i, 'error': f'File load error: {e}', 'raw_response': None})
            continue

    prompt = INSTRUCTIONS + json.dumps({'documents': batch}, ensure_ascii=False, indent=2)
    print(len(prompt))
    print(f"Processing batch {i} with {len(batch)} documents...")

    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    config = types.GenerateContentConfig(response_mime_type="text/plain")

    try:
        response_text = ''.join(
            chunk.text for chunk in client.models.generate_content_stream(
                model=MODEL_NAME, contents=contents, config=config
            )
        )

        # Clean up potential Markdown code fences
        cleaned = response_text.strip()
        cleaned = re.sub(r"^```json", "", cleaned, flags=re.MULTILINE)
        cleaned = re.sub(r"```$", "", cleaned, flags=re.MULTILINE)
        cleaned = cleaned.strip()

        data = json.loads(cleaned)
        results = data.get('results', [])
        print(f"Batch {i}: parsed {len(results)} results")
        aggregated_results.extend(results)

    except json.JSONDecodeError as e:
        print(f"Batch {i} JSON parse error: {e}\nIncluding raw response in results.json")
        aggregated_results.append({'batch': i, 'error': str(e), 'raw_response': cleaned})

    except Exception as e:
        print(f"Batch {i}: onverwachte fout: {e}")
        aggregated_results.append({'batch': i, 'error': str(e), 'raw_response': None})

# Save final aggregated results
output_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/events/results_events_batch_2001_tm_3813.json'
with open(output_path, 'w', encoding='utf-8') as outfile:
    json.dump({'results': aggregated_results}, outfile, ensure_ascii=False, indent=2)

print(f"Saved aggregated results: {len(aggregated_results)} entries to '{output_path}'")


In [17]:
import json
import csv
import re
import os
from google import genai
from google.genai import types

# API setup
client = genai.Client(api_key=API_key)
MODEL_NAME = "models/gemini-2.0-flash"
print(f"Using model: {MODEL_NAME} for API generation")

# Jouw document_id lijst
documentids = ["17-443", "195-353"
]

# Batch directory (waar alle batch bestanden staan)
BATCH_DIR = "/home/nena-meijer/PyCharmMiscProject/information_extraction/batches"

# Output bestand
output_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/events/filtered_results_last_docs.json'

# Instruction prompt (dezelfde als eerder)
INSTRUCTIONS = (
    "Je krijgt een JSON-array met documenten met betrekking tot de coronacrisis in het Nederlands of Engels.\n"
    "Geef de resultaten terug in het Nederlands. Voor elk document extraheer je de volgende informatie:\n"
    "1. events: alle events met betrekking tot de coronacrisis en de besluitvorming rondom de coronacrisis. Geef de bijbehorende datums van de events in het format: YYYY-MM-DD. Een 'event' is een gebeurtenis die directe invloed heeft op de reactie op de coronacrisis, beleidsbeslissingen, of significante veranderingen in de situatie. Voorbeelden zijn: aankondigingen van nieuwe maatregelen, wetenschappelijke publicaties die het beleid beïnvloeden, belangrijke politieke debatten, en wijzigingen in test- of vaccinatiestrategieën.\n"
    "De relatieve datums (morgen, gister, vandaag, volgende week, etc.) moeten ook worden omgezet naar het formaat YYYY-MM-DD.  Het is cruciaal dat alle datums, inclusief relatieve datums en verwijzingen naar dagen van de week, correct worden omgezet naar het YYYY-MM-DD formaat. Gebruik de context van het document om de juiste datum te bepalen. Indien de documenten datums bevatten gerelateerd aan 'vandaag' interpreteer dit als de datum van analyse, welke vandaag: 2020-06-22 is. Alle datums, absoluut en relatief, moeten geretourneerd worden in het format YYYY-MM-DD. Er mogen geen uitzonderingen zijn.\n"
    "Geef een beschrijving van het event die voldoende context bevat om te begrijpen wat er gebeurt en waarom het relevant is voor de coronacrisis.  Streef naar een beschrijving van minimaal 20 woorden, maar wees niet bang om meer te gebruiken als dat nodig is om de context volledig te omvatten.  Bij het beschrijven van een event, probeer de volgende vragen te beantwoorden: Wie was erbij betrokken? Wat is er precies gebeurd? Waar vond het plaats (indien relevant)? Waarom is dit event belangrijk in de context van de coronacrisis?\n"
    "De beschrijving van het event moet gebaseerd zijn op de omliggende tekst in het document. Probeer zinnen of fragmenten uit het document te gebruiken om de beschrijving zo accuraat en informatief mogelijk te maken. Vermijd vage of onvolledige beschrijvingen zoals 'Gesprek gehad' of 'Beslissing genomen'. Geef altijd de belangrijkste details.\n"
    "Hier zijn een paar voorbeelden:\n"
    "INPUT DOCUMENT: 'Vandaag heeft het RIVM nieuwe richtlijnen gepubliceerd voor het gebruik van mondkapjes in het openbaar vervoer.  Morgen zal de minister een persconferentie geven om de details toe te lichten.'\n"
    "OUTPUT: {\"document_id\": \"voorbeeld-1\", \"events\": [{\"event\": \"Het RIVM publiceert nieuwe richtlijnen voor het gebruik van mondkapjes in het openbaar vervoer, waardoor reizigers en vervoerders nieuwe regels moeten volgen.\", \"date\": \"2023-10-27\"}, {\"event\": \"De minister geeft een persconferentie over de nieuwe RIVM-richtlijnen voor mondkapjes in het openbaar vervoer, waarin de details en de achtergrond van de regels worden toegelicht.\", \"date\": \"2023-10-28\"}]}\n"
    "Stuur als output één JSON-object met een \"results\"-array.\n"
    "Neem per document het document_id mee in de response.\n"
    "Lever uitsluitend geldige JSON zonder extra markdown‑fences, zonder trailing commas, met alle strings correct geescaped.\n"
)

# 🔥 Load alle batch-bestanden en verzamel de juiste document_id's
filtered_docs = []

for filename in os.listdir(BATCH_DIR):
    if not filename.endswith('.json'):
        continue

    with open(os.path.join(BATCH_DIR, filename), 'r', encoding='utf-8') as f:
        try:
            batch = json.load(f)
        except json.JSONDecodeError as e:
            print(f"Fout bij laden {filename}: {e}")
            continue

        if isinstance(batch, dict) and 'documents' in batch:
            for doc in batch['documents']:
                if isinstance(doc, dict) and doc.get('id') in documentids:
                    filtered_docs.append(doc)
        else:
            print(f"{filename}: bevat geen geldige 'documents' lijst, overslaan...")

print(f"Totaal gefilterde documenten: {len(filtered_docs)}")


# Maak prompt per document
aggregated_results = []

for doc in filtered_docs:
    prompt_data = {'documents': [doc]}  # 1 document per keer
    prompt = INSTRUCTIONS + json.dumps(prompt_data, ensure_ascii=False, indent=2)

    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    config = types.GenerateContentConfig(response_mime_type="text/plain")

    try:
        response_text = ''.join(
            chunk.text for chunk in client.models.generate_content_stream(
                model=MODEL_NAME, contents=contents, config=config
            )
        )

        # Opschonen van response
        cleaned = response_text.strip()
        cleaned = re.sub(r"^```json", "", cleaned, flags=re.MULTILINE)
        cleaned = re.sub(r"```$", "", cleaned, flags=re.MULTILINE)
        cleaned = cleaned.strip()

        data = json.loads(cleaned)
        results = data.get('results', [])
        print(f"Document {doc['id']}: parsed {len(results)} results")
        aggregated_results.extend(results)

    except json.JSONDecodeError as e:
        print(f"Document {doc['id']} JSON parse error: {e}")
        aggregated_results.append({'id': doc['id'], 'error': str(e), 'raw_response': cleaned})

    except Exception as e:
        print(f"Document {doc['id']}: onverwachte fout: {e}")
        aggregated_results.append({'id': doc['id'], 'error': str(e), 'raw_response': None})

# Save alle resultaten
with open(output_path, 'w', encoding='utf-8') as outfile:
    json.dump({'results': aggregated_results}, outfile, ensure_ascii=False, indent=2)

print(f"Saved filtered results: {len(aggregated_results)} entries to '{output_path}'")


Using model: models/gemini-2.0-flash for API generation
Totaal gefilterde documenten: 1
Document 195-353 JSON parse error: Unterminated string starting at: line 319 column 20 (char 26708)
Saved filtered results: 1 entries to '/home/nena-meijer/PyCharmMiscProject/information_extraction/events/filtered_results_last_docs.json'


In [ ]:
import json
import csv
import time

# Paden naar je JSON-bestanden
json_files = [
    '/home/nena-meijer/PyCharmMiscProject/information_extraction/events/results_events_batch_1_tm_500.json',
    '/home/nena-meijer/PyCharmMiscProject/information_extraction/events/results_events_batch_501_tm_1500.json',
    '/home/nena-meijer/PyCharmMiscProject/information_extraction/events/results_events_batch_1501_tm_2000.json',
    '/home/nena-meijer/PyCharmMiscProject/information_extraction/events/results_events_batch_2001_tm_3813.json'
]

# CSV output pad
csv_output = '/home/nena-meijer/PyCharmMiscProject/database/Event.csv'

start_time = time.time()

all_results = []

# Lees en voeg alleen items met document_id samen (skip raw_response!)
for file in json_files:
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        for item in data['results']:
            if 'document_id' in item:
                all_results.append(item)
            elif 'raw_response' in item:
                # Sla deze over
                continue

print(f"Totaal bruikbare items zonder raw_response: {len(all_results)}")

# Schrijf naar CSV
with open(csv_output, 'w', encoding='utf-8', newline='') as f_csv:
    writer = csv.writer(f_csv)
    writer.writerow(['document_id', 'date', 'description'])  # header

    count = 0
    for item in all_results:
        document_id = item.get('document_id')
        if not document_id:
            continue
        for event in item.get('events', []):
            date = event.get('date', '')
            description = event.get('event', '')
            writer.writerow([document_id, date, description])
            count += 1

print(f"Totaal geschreven rijen: {count}")
print(f"CSV succesvol opgeslagen als: {csv_output}")
print(f"Totale duur: {time.time() - start_time:.2f} seconden")

In [18]:
import json
import pandas as pd

# Pad naar je bestanden
json_path = '/home/nena-meijer/PyCharmMiscProject/information_extraction/events/filtered_results.json'
csv_path = '/home/nena-meijer/PyCharmMiscProject/database/Event.csv'

# JSON-bestand inlezen
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Events extraheren
events_list = []
for doc in data['results']:
    document_id = doc['document_id']
    for event in doc['events']:
        events_list.append({
            'document_id': document_id,
            'date': event['date'],
            'description': event['event']
        })

# Nieuwe DataFrame van events
new_events_df = pd.DataFrame(events_list)

# Bestaande CSV inlezen
existing_events_df = pd.read_csv(csv_path)

# Samenvoegen
combined_df = pd.concat([existing_events_df, new_events_df], ignore_index=True)

# Optioneel: duplicaten verwijderen op basis van alle kolommen
combined_df.drop_duplicates(inplace=True)

# Opslaan naar CSV
combined_df.to_csv(csv_path, index=False)

print(f"Toegevoegd {len(new_events_df)} nieuwe events aan {csv_path}.")


Toegevoegd 400 nieuwe events aan /home/nena-meijer/PyCharmMiscProject/database/Event.csv.
